In [71]:
import pandas as pd 
import os
from scripts.parser_cotahist import ler_cotahist

path = ("dados_brutos/COTAHIST_A2024.TXT")

df = ler_cotahist(path, nrows=100000)
df.head()

,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume
0,00,COTAHIST,024BOVESPA 2,1231,NaN,NaN,NaN
1,01,20240102,AALR3,ALLIAR,1020.0,850.0,4.014875e+08
2,01,20240102,ABCB4,ABC BRASIL,2398.0,2288.0,4.494731e+09
3,01,20240102,ABEV3,AMBEV S/A,1372.0,1371.0,1.598391e+10
4,01,20240102,BBDC3,BRADESCO,1526.0,1511.0,6.857685e+09


In [72]:
#salvar o df em formato parquet
os.makedirs("bronze", exist_ok=True)

df.to_parquet("bronze/cotahist_2024.parquet", index=False)
print("O arquivo Parquet foi criado em: bronze/cotahist_2024.parquet")

O arquivo Parquet foi criado em: bronze/cotahist_2024.parquet


In [73]:
#dividir por 100 conforme manual da B3 para tratar decimais
df["preco_abertura"] = df["preco_abertura"] / 100
df["preco_fechamento"] = df["preco_fechamento"] / 100

#filtrar somento por acoes 
df = df[df["tipo_registro"] == "01"]

#converter para formato de datetime (sem o filtro de acoes estava pegando o cabeçalho e estava retornando erro)
df["data"] = pd.to_datetime(df["data"], format="%Y%m%d")

#Limpar espaços vazios nos textos
df["codigo_acao"] = df["codigo_acao"].str.strip()
df["nome_empresa"] = df["nome_empresa"].str.strip()


In [74]:
#salvar na camada silver
os.makedirs("silver", exist_ok=True)
df.to_parquet("silver/cotahist_2024_silver.parquet", index=False)
print("Dados limpos e tipados, camada silver concluida!")

df.head()

Dados limpos e tipados, camada silver concluida!


,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume
1,01,2024-01-02,AALR3,ALLIAR,10.20,8.50,4.014875e+08
2,01,2024-01-02,ABCB4,ABC BRASIL,23.98,22.88,4.494731e+09
3,01,2024-01-02,ABEV3,AMBEV S/A,13.72,13.71,1.598391e+10
4,01,2024-01-02,BBDC3,BRADESCO,15.26,15.11,6.857685e+09
5,01,2024-01-02,ALPA3,ALPARGATAS,10.11,10.00,2.912000e+06


In [75]:
df = df.sort_values(['codigo_acao','data'])
df['variacao_diaria'] = df.groupby('codigo_acao')[
    'preco_fechamento'].pct_change() * 100


df.head(10)

,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume,variacao_diaria
772,01,2024-01-02,5GTK11,INVESTO 5GTK,82.58,81.39,587130.0,NaN
2489,01,2024-01-03,5GTK11,INVESTO 5GTK,81.39,80.10,265970.0,-1.584961
4205,01,2024-01-04,5GTK11,INVESTO 5GTK,80.00,79.47,4250700.0,-0.786517
5927,01,2024-01-05,5GTK11,INVESTO 5GTK,79.47,79.34,158810.0,-0.163584
7678,01,2024-01-08,5GTK11,INVESTO 5GTK,79.34,81.47,543920.0,2.684648
9365,01,2024-01-09,5GTK11,INVESTO 5GTK,81.47,81.95,114270.0,0.589174
11035,01,2024-01-10,5GTK11,INVESTO 5GTK,79.75,81.93,3404700.0,-0.024405
12717,01,2024-01-11,5GTK11,INVESTO 5GTK,81.00,81.79,2423730.0,-0.170878
14441,01,2024-01-12,5GTK11,INVESTO 5GTK,81.79,81.55,840790.0,-0.293434
16108,01,2024-01-15,5GTK11,INVESTO 5GTK,81.56,81.60,570930.0,0.061312


In [80]:
#calcular Média Móvel de 21 dias (mais usada pelo mercado)
df['media_movel_21'] = df.groupby('codigo_acao')[
    'preco_fechamento'].transform(
        lambda x: x.rolling(window=21).mean())

df[df['media_movel_21'].notna()].head()

,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume,variacao_diaria,media_movel_21
34652,01,2024-01-30,5GTK11,INVESTO 5GTK,88.40,86.30,23192700.0,-1.213370,83.631905
36413,01,2024-01-31,5GTK11,INVESTO 5GTK,85.86,84.84,3098020.0,-1.691773,83.796190
38115,01,2024-02-01,5GTK11,INVESTO 5GTK,84.84,84.79,431720.0,-0.058934,84.019524
39918,01,2024-02-02,5GTK11,INVESTO 5GTK,86.33,86.28,34520.0,1.757283,84.343810
41628,01,2024-02-05,5GTK11,INVESTO 5GTK,87.39,86.53,10767150.0,0.289754,84.686190
